# NB-17: Pipeline Orchestration -- Full Inference Flow with VACE and S2V Dispatch

## Learning Objectives
- Trace the complete `WanVideoPipeline.__call__` flow from preprocessing through denoising to VAE decode
- Understand how `model_fn_wan_video` dispatches between VACE, S2V, and base DiT paths
- See classifier-free guidance (CFG) applied within the denoising loop with `FlowMatchScheduler` step updates

## Prerequisites
- **Prior notebooks:** NB-07 (WanModel forward pass -- 8-step pipeline from patchify to unpatchify), NB-10 (VAE Decoder -- latent-to-video reconstruction with temporal formula T_out = T_lat*4 - 3), NB-12 (VACE Pipeline Integration -- hint pre-computation and additive injection at even DiT layers), NB-16 (WanS2VModel forward pass -- 7-stage trace with unified sequence composition)
- **Assumed concepts:** DiT block loop (NB-07), VACE hint injection (NB-12), S2V dual-timestep and audio injection (NB-16), VAE decode shape arithmetic (NB-10)

## Concept Map
- This notebook shows the **production pipeline orchestration**: how `WanVideoPipeline.__call__` connects scheduler, preprocessing units, model dispatch, CFG, and VAE decode
- The **PipelineUnit** pattern provides modular preprocessing; we focus on the VACE and S2V units
- The **three-way dispatch** in `model_fn_wan_video` routes to S2V (priority 1), VACE (priority 2), or base DiT (priority 3)
- This is the **final notebook** of the course -- after this, you understand the complete WAN 2.2 architecture

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, wan_video_vace.py, wan_video_dit_s2v.py, flow_match.py (setup)
import sys, types, importlib.util, pathlib
import torch
import torch.nn as nn
import torch.nn.functional as F

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for pipeline demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load modules in dependency order
# 1. wan_video_dit.py (base DiT -- dependency of VACE and S2V)
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit_mod
_spec.loader.exec_module(dit_mod)

# 2. wan_video_vace.py (VACE control encoder)
_vace_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_vace.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_vace", _vace_path)
vace_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_vace"] = vace_mod
_spec.loader.exec_module(vace_mod)

# 3. wan_video_dit_s2v.py (S2V model)
_s2v_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit_s2v.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit_s2v", _s2v_path)
s2v_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit_s2v"] = s2v_mod
_spec.loader.exec_module(s2v_mod)

# 4. flow_match.py (FlowMatchScheduler -- minimal deps: torch, math, typing_extensions)
_fm_path = PROJECT_ROOT / "diffsynth" / "diffusion" / "flow_match.py"
_spec = importlib.util.spec_from_file_location("diffsynth.diffusion.flow_match", _fm_path)
fm_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(fm_mod)

# Import key classes
WanModel = dit_mod.WanModel
sinusoidal_embedding_1d = dit_mod.sinusoidal_embedding_1d
precompute_freqs_cis_3d = dit_mod.precompute_freqs_cis_3d
VaceWanModel = vace_mod.VaceWanModel
WanS2VModel = s2v_mod.WanS2VModel
FlowMatchScheduler = fm_mod.FlowMatchScheduler

from einops import rearrange

print("Setup complete. Loaded: WanModel, VaceWanModel, WanS2VModel, FlowMatchScheduler")

## WAN 2.2 Full Pipeline Orchestration (NB-17 Scope)

```
WAN 2.2 Full Pipeline Orchestration
====================================================

ENTRY: WanVideoPipeline.__call__(prompt, [vace_video], [audio], ...)
  |
  v
PHASE 1: SCHEDULER SETUP
  scheduler.set_timesteps(num_inference_steps=N, shift=5.0)
  -> sigmas: [N] shifted schedule, timesteps: [N] = sigmas * 1000
  Source: flow_match.py, lines 30-39, 137-143
  |
  v
PHASE 2: INPUT PREPARATION (PipelineUnit chain)
  inputs_shared, inputs_posi, inputs_nega = dicts
  for unit in self.units:
      unit_runner(unit, pipe, shared, posi, nega)
      |
      +-- ShapeChecker    -> height, width, num_frames (aligned)
      +-- NoiseInitializer -> noise [B, 16, T_lat, H/8, W/8]
      +-- PromptEmbedder  -> context [B, L, 512] (separate posi/nega)
      +-- S2V Unit        -> audio_embeds, motion_latents, pose_latents (if S2V)
      +-- VACE Unit       -> vace_context [B, vace_in_dim, T_lat, H/8, W/8] (if VACE)
      +-- CfgMerger       -> batch-merges posi+nega (if cfg_merge=True)
      +-- ... (other units skipped: animate, VAP, LongCat, etc.)
  Source: wan_video.py, lines 284-286
  |
  v
PHASE 3: DENOISING LOOP
  for timestep in scheduler.timesteps:
      noise_pred = model_fn_wan_video(...)  <-- three-way dispatch
      CFG: noise_pred = nega + cfg_scale * (posi - nega)
      latents = scheduler.step(noise_pred, timestep, latents)
  Source: wan_video.py, lines 290-314
  |
  v
PHASE 4: POST-PROCESSING
  WanVideoPostUnit_S2V: cat motion frames back (if S2V + motion video)
  Source: wan_video.py, lines 324-325, 895-903
  |
  v
PHASE 5: VAE DECODE
  video = vae.decode(latents)
  [1, 16, T_lat, H/8, W/8] -> [1, 3, T_out, H, W]  (T_out = T_lat * 4 - 3)
  Source: wan_video.py, line 328; wan_video_vae.py
```

## The PipelineUnit Runner Pattern

Source: `diffsynth/diffusion/base_pipeline.py`, lines 13-57 (PipelineUnit), lines 421-451 (PipelineUnitRunner)

Before the denoising loop, `WanVideoPipeline.__call__` iterates through `self.units` -- a list of 22 `PipelineUnit` subclasses. Each unit declares its input/output parameters and one of three processing modes:

| Mode | Flag | Behavior | Example Units |
|------|------|----------|---------------|
| Standard | (default) | Extract params from `inputs_shared`, call `process()`, update `inputs_shared` | ShapeChecker, NoiseInitializer |
| Separate CFG | `seperate_cfg=True` | Call `process()` twice -- once for positive prompt, once for negative | PromptEmbedder |
| Take Over | `take_over=True` | Receives all three dicts (`shared`, `posi`, `nega`) directly | S2V Unit, VACE Unit, CfgMerger |

### Simplified PipelineUnitRunner Logic

```python
# Source: diffsynth/diffusion/base_pipeline.py, lines 421-451
class PipelineUnitRunner:
    def __call__(self, unit, pipe, inputs_shared, inputs_posi, inputs_nega):
        if unit.take_over:
            return unit.process(pipe, inputs_shared=inputs_shared,
                                inputs_posi=inputs_posi, inputs_nega=inputs_nega)
        elif unit.seperate_cfg:
            # Call process for positive, then negative
            posi_out = unit.process(pipe, **posi_inputs)
            nega_out = unit.process(pipe, **nega_inputs)
            ...
        else:
            # Standard: extract from shared, call, update shared
            params = {name: inputs_shared.get(name) for name in unit.input_params}
            outputs = unit.process(pipe, **params)
            inputs_shared.update(outputs)
        return inputs_shared, inputs_posi, inputs_nega
```

### Full Unit List

Source: `diffsynth/pipelines/wan_video.py`, lines 55-83

```python
self.units = [
    WanVideoUnit_ShapeChecker(),       # Standard
    WanVideoUnit_NoiseInitializer(),   # Standard
    WanVideoUnit_PromptEmbedder(),     # seperate_cfg=True
    WanVideoUnit_S2V(),                # take_over=True
    WanVideoUnit_InputVideoEmbedder(), # Standard
    WanVideoUnit_ImageEmbedderVAE(),   # Standard
    WanVideoUnit_ImageEmbedderCLIP(),  # Standard
    WanVideoUnit_ImageEmbedderFused(), # Standard
    WanVideoUnit_FunControl(),         # Standard
    WanVideoUnit_FunReference(),       # Standard
    WanVideoUnit_FunCameraControl(),   # Standard
    WanVideoUnit_SpeedControl(),       # Standard
    WanVideoUnit_VACE(),               # take_over=True
    WanVideoUnit_AnimateVideoSplit(),  # Standard
    WanVideoUnit_AnimatePoseLatents(), # Standard
    WanVideoUnit_AnimateFacePixelValues(), # Standard
    WanVideoUnit_AnimateInpaint(),     # Standard
    WanVideoUnit_VAP(),                # Standard
    WanVideoUnit_UnifiedSequenceParallel(), # Standard
    WanVideoUnit_TeaCache(),           # Standard
    WanVideoUnit_CfgMerger(),          # take_over=True
    WanVideoUnit_LongCatVideo(),       # Standard
]
```

We focus on **WanVideoUnit_VACE** and **WanVideoUnit_S2V** -- the two paths studied in Phases 4 and 5. The other 20 units handle features outside this course's scope (image animation, FunControl, LongCat video, etc.).

In [ ]:
# Source: diffsynth/diffusion/base_pipeline.py, lines 421-451 (simplified)
# Demonstrate the three routing modes of PipelineUnitRunner

# Mode 1: Standard -- extracts from shared, updates shared
inputs_shared = {"height": 32, "width": 32, "num_frames": 5}
print("Mode 1 -- Standard (e.g., ShapeChecker):")
print(f"  Input: {inputs_shared}")
# ShapeChecker aligns dimensions and computes latent temporal dim
T_lat = (inputs_shared["num_frames"] - 1) // 4 + 1  # temporal compression
inputs_shared["latent_temporal"] = T_lat
print(f"  Output: {inputs_shared}")
assert inputs_shared["latent_temporal"] == 2, f"Expected T_lat=2, got {T_lat}"

# Mode 2: Separate CFG -- calls process for positive AND negative
inputs_posi = {}
inputs_nega = {}
print("\nMode 2 -- Separate CFG (e.g., PromptEmbedder):")
# Positive: real prompt embedding
inputs_posi["context"] = torch.randn(1, 77, 512)  # text_dim=512 reduced
# Negative: empty/null prompt embedding
inputs_nega["context"] = torch.randn(1, 77, 512)
print(f"  Positive context: {inputs_posi['context'].shape}")
print(f"  Negative context: {inputs_nega['context'].shape}")
assert inputs_posi["context"].shape == torch.Size([1, 77, 512])
assert inputs_nega["context"].shape == torch.Size([1, 77, 512])

# Mode 3: Take-over -- unit gets all three dicts
print("\nMode 3 -- Take-over (e.g., VACE/S2V units):")
print("  Unit receives inputs_shared, inputs_posi, inputs_nega directly")
print("  Can modify any dict -- used for complex preprocessing")
print("\nPipelineUnit routing demo complete.")

## WanVideoUnit_VACE Preprocessing

Source: `diffsynth/pipelines/wan_video.py`, lines 621-681

The VACE unit (`take_over=True`) constructs `vace_context` from a control video and mask. It:
1. Splits the video into **inactive** (masked-out) and **reactive** (visible) regions
2. VAE-encodes each to latent space (we simulate with synthetic tensors)
3. Concatenates along the channel dimension
4. Downsamples the mask to latent spatial resolution
5. Produces `vace_context` with shape `[B, vace_in_dim, T_lat, H_lat, W_lat]`

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, lines 636-678
# VACE preprocessing walkthrough -- simulating WanVideoUnit_VACE.process()

# Reduced config (matching Phase 4: NB-12)
B = 1
num_frames = 5
H, W = 32, 32
z_dim = 16  # VAE latent channels
T_lat = (num_frames - 1) // 4 + 1  # = 2
H_lat, W_lat = H // 8, W // 8  # = 4, 4

print("VACE Preprocessing (WanVideoUnit_VACE)")
print("=" * 50)

# Step 1: Construct inactive/reactive from mask
# Source: wan_video.py, lines 636-644
print(f"\nStep 1: Split video into inactive/reactive regions")
print(f"  Input video: [{B}, 3, {num_frames}, {H}, {W}]")
print(f"  Input mask:  [{B}, 3, {num_frames}, {H}, {W}]")
print(f"  inactive = video * (1 - mask)  -- masked-out regions")
print(f"  reactive = video * mask         -- visible regions")

# Step 2: VAE-encode each (simulated with synthetic latents)
# Source: wan_video.py, lines 650-660
print(f"\nStep 2: VAE-encode to latent space (simulated)")
inactive_latents = torch.randn(B, z_dim, T_lat, H_lat, W_lat)
reactive_latents = torch.randn(B, z_dim, T_lat, H_lat, W_lat)
print(f"  inactive_latents: {inactive_latents.shape}  [B, z_dim={z_dim}, T_lat={T_lat}, H_lat={H_lat}, W_lat={W_lat}]")
print(f"  reactive_latents: {reactive_latents.shape}")
assert inactive_latents.shape == torch.Size([B, z_dim, T_lat, H_lat, W_lat])
assert reactive_latents.shape == torch.Size([B, z_dim, T_lat, H_lat, W_lat])

# Step 3: Concatenate along channel dim
# Source: wan_video.py, lines 662-665
print(f"\nStep 3: Concatenate along channel dim")
vace_video_latents = torch.cat([inactive_latents, reactive_latents], dim=1)
print(f"  cat([inactive, reactive], dim=1): {vace_video_latents.shape}")
assert vace_video_latents.shape == torch.Size([B, z_dim * 2, T_lat, H_lat, W_lat])

# Step 4: Downsample mask to latent spatial resolution
# Source: wan_video.py, lines 667-675
# Mask uses spatial chunking with P=8, Q=8, producing 64 channels from 1 mask channel
print(f"\nStep 4: Downsample mask to latent resolution")
vace_mask_latents = torch.randn(B, 64, T_lat, H_lat, W_lat)
print(f"  Mask spatial chunking (P=8, Q=8): 1 channel -> 64 channels")
print(f"  vace_mask_latents: {vace_mask_latents.shape}")
assert vace_mask_latents.shape == torch.Size([B, 64, T_lat, H_lat, W_lat])

# Step 5: Concatenate video latents + mask latents -> vace_context
# Source: wan_video.py, line 678
print(f"\nStep 5: Construct vace_context")
vace_in_dim = z_dim * 2 + 64  # 16 + 16 + 64 = 96
vace_context_tensor = torch.cat([vace_video_latents, vace_mask_latents], dim=1)
print(f"  cat([video_latents, mask_latents], dim=1): {vace_context_tensor.shape}")
print(f"  vace_in_dim = {z_dim}(inactive) + {z_dim}(reactive) + 64(mask) = {vace_in_dim}")
assert vace_context_tensor.shape == torch.Size([B, vace_in_dim, T_lat, H_lat, W_lat])

print(f"\nFinal vace_context: {vace_context_tensor.shape}")
print(f"  This feeds into VaceWanModel.forward() as shown in NB-12")

## WanVideoUnit_S2V Preprocessing

Source: `diffsynth/pipelines/wan_video.py`, lines 811-892

The S2V unit (`take_over=True`) prepares three conditioning signals:
1. **audio_embeds**: WAV2Vec features -> CausalAudioEncoder -> weighted aggregation -> `[B, T_audio, audio_dim]`
2. **motion_latents**: Motion video -> VAE encode -> `[B, 16, T_motion_lat, H_lat, W_lat]`
3. **pose_latents**: Pose video -> VAE encode -> `[B, 16, T_lat, H_lat, W_lat]`

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, lines 811-881
# S2V preprocessing walkthrough -- simulating WanVideoUnit_S2V.process()

# Reduced config (matching Phase 5: NB-16)
B = 1
num_frames = 5
H, W = 32, 32
z_dim = 16
T_lat = (num_frames - 1) // 4 + 1  # = 2
H_lat, W_lat = H // 8, W // 8  # = 4, 4
audio_dim = 384
num_wav2vec_layers = 25
video_length = 16  # audio temporal length

print("S2V Preprocessing (WanVideoUnit_S2V)")
print("=" * 50)

# Step 1: Audio embeddings
# Source: wan_video.py, lines 828-835
print(f"\nStep 1: Audio embedding preparation")
print(f"  Raw WAV2Vec features: [{B}, {num_wav2vec_layers}, {audio_dim}, {video_length}]")
print(f"  -> CausalAudioEncoder (4x temporal downsample, layer aggregation)")
audio_embeds = torch.randn(B, num_wav2vec_layers, audio_dim, video_length)
print(f"  audio_embeds (raw input): {audio_embeds.shape}")
assert audio_embeds.shape == torch.Size([B, num_wav2vec_layers, audio_dim, video_length])

# Step 2: Motion latents
# Source: wan_video.py, lines 841-855
print(f"\nStep 2: Motion video -> VAE encode (simulated)")
motion_frames = 5  # available motion frames from previous clip
motion_latents_list = [torch.randn(z_dim, motion_frames, H_lat * 2, W_lat * 2)]
print(f"  Motion video frames: {motion_frames}")
print(f"  motion_latents: list of 1 tensor, shape {motion_latents_list[0].shape}")
print(f"  Note: spatial dims are at original resolution before patch downsampling")
assert motion_latents_list[0].shape == torch.Size([z_dim, motion_frames, H_lat * 2, W_lat * 2])

# Step 3: Pose latents
# Source: wan_video.py, lines 860-875
F_content = num_frames - 1  # content frames (excluding ref frame)
print(f"\nStep 3: Pose skeleton -> VAE encode (simulated)")
pose_latents = torch.randn(B, z_dim, F_content, H_lat * 2, W_lat * 2)
print(f"  pose_latents: {pose_latents.shape}")
print(f"  Note: pose covers content frames only (not reference frame)")
assert pose_latents.shape == torch.Size([B, z_dim, F_content, H_lat * 2, W_lat * 2])

print(f"\nS2V preprocessing output summary:")
print(f"  audio_embeds:    {audio_embeds.shape}  (raw WAV2Vec features for CausalAudioEncoder)")
print(f"  motion_latents:  list of 1 tensor, {motion_latents_list[0].shape}")
print(f"  pose_latents:    {pose_latents.shape}")
print(f"  These feed into WanS2VModel.forward() as shown in NB-16")

## FlowMatchScheduler API

Source: `diffsynth/diffusion/flow_match.py`, lines 5-39 (class), lines 137-158 (set_timesteps + step)

The `FlowMatchScheduler` controls the denoising loop by providing a sigma schedule and a step formula. For WAN 2.2:
- **Sigma schedule:** `sigmas = shift * sigmas / (1 + (shift - 1) * sigmas)` with `shift=5.0`
- **Step formula:** `prev_sample = sample + noise_pred * (sigma_next - sigma_current)`
- **Timesteps:** `timesteps = sigmas * 1000`

We use 5 steps per D-06 (production default is 50).

In [ ]:
# Source: diffsynth/diffusion/flow_match.py, lines 30-39, 149-158
scheduler = FlowMatchScheduler("Wan")
scheduler.set_timesteps(num_inference_steps=5, shift=5.0)

print("FlowMatchScheduler API Demo")
print("=" * 50)

print(f"\nSigma schedule (Wan template, shift=5.0, 5 steps):")
for i, (sigma, t) in enumerate(zip(scheduler.sigmas, scheduler.timesteps)):
    print(f"  Step {i}: sigma={sigma:.4f}, timestep={t:.1f}")

assert len(scheduler.timesteps) == 5, f"Expected 5 timesteps, got {len(scheduler.timesteps)}"
assert scheduler.sigmas[0] > scheduler.sigmas[-1], "Sigmas should decrease (high noise -> low noise)"
assert all(scheduler.sigmas[i] >= scheduler.sigmas[i+1] for i in range(len(scheduler.sigmas)-1)), \
    "Sigmas must be monotonically decreasing"

# Demonstrate one step of the step formula
# Source: flow_match.py, lines 149-158
sample = torch.randn(1, 16, 2, 4, 4)  # latent noise [B, C, T_lat, H_lat, W_lat]
noise_pred = torch.randn_like(sample)   # model output (noise prediction)
prev_sample = scheduler.step(noise_pred, scheduler.timesteps[0], sample)
assert prev_sample.shape == sample.shape, f"Step output shape mismatch: {prev_sample.shape} != {sample.shape}"

print(f"\nStep formula demo:")
print(f"  sample:      {sample.shape}")
print(f"  noise_pred:  {noise_pred.shape}")
print(f"  prev_sample: {prev_sample.shape}")
print(f"\n  Formula: prev = sample + noise_pred * (sigma_next - sigma_current)")
print(f"  sigma_current = {scheduler.sigmas[0]:.4f}")
print(f"  sigma_next    = {scheduler.sigmas[1]:.4f}")
print(f"  delta_sigma   = {(scheduler.sigmas[1] - scheduler.sigmas[0]):.4f}")

# Verify step formula manually
expected = sample + noise_pred * (scheduler.sigmas[1] - scheduler.sigmas[0])
assert torch.allclose(prev_sample, expected, atol=1e-5), "Step formula mismatch"
print(f"\n  Manual verification: prev = sample + noise_pred * delta_sigma -- MATCHES")

## Model Instantiation (Reduced Config)

We instantiate all three model variants with the reduced configs used throughout this course. These models will be called by `model_fn_wan_video` in the denoising loop (Plan 02).

| Model | Config Source | Key Params |
|-------|-------------|------------|
| WanModel (base DiT) | Phase 2 (NB-07) | dim=384, num_heads=4, num_layers=5 |
| VaceWanModel | Phase 4 (NB-12) | 3 blocks, vace_layers=(0,2,4), vace_in_dim=96 |
| WanS2VModel | Phase 5 (NB-16) | audio_inject_layers=[0,2], zip_frame_buckets=[1,2,16] |

In [ ]:
# Source: wan_video_dit.py, line 273 (WanModel)
# Source: wan_video_vace.py, line 27 (VaceWanModel)
# Source: wan_video_dit_s2v.py, line 359 (WanS2VModel)

# Shared reduced config
dim = 384
num_heads = 4
head_dim = dim // num_heads  # 96
ffn_dim = 1024
in_dim = 16
out_dim = 16
text_dim = 512
freq_dim = 256
eps = 1e-6
patch_size = (1, 2, 2)
num_layers = 5

# --- 1. WanModel (base DiT) ---
dit = WanModel(
    dim=dim, in_dim=in_dim, ffn_dim=ffn_dim, out_dim=out_dim,
    text_dim=text_dim, freq_dim=freq_dim, eps=eps,
    patch_size=patch_size, num_heads=num_heads, num_layers=num_layers,
    has_image_input=False,
)
dit.eval()

# Apply patchify monkey-patch (same fix as NB-07)
# Source: wan_video_dit.py, line 340 (bug: returns tensor only, not tuple)
# Source: wan_video_dit.py, line 375 (forward expects: x, (f, h, w) = self.patchify(x))
import types as _types

def patchify_fixed(self, x, control_camera_latents_input=None):
    """Corrected patchify: Conv3d + flatten + return grid dims."""
    x = self.patch_embedding(x)
    if self.control_adapter is not None and control_camera_latents_input is not None:
        y_camera = self.control_adapter(control_camera_latents_input)
        x = [u + v for u, v in zip(x, y_camera)]
        x = x[0].unsqueeze(0)
    b, c, f, h, w = x.shape
    x = rearrange(x, 'b c f h w -> b (f h w) c')
    return x, (f, h, w)

dit.patchify = _types.MethodType(patchify_fixed, dit)

dit_params = sum(p.numel() for p in dit.parameters())
print(f"1. WanModel (base DiT):  {dit_params:,} params, {num_layers} blocks")
assert hasattr(dit, 'blocks') and len(dit.blocks) == num_layers
assert hasattr(dit, 'patch_embedding')
assert hasattr(dit, 'head')

# --- 2. VaceWanModel ---
vace_layers = (0, 2, 4)
vace_in_dim = 96  # 16(inactive) + 16(reactive) + 64(mask)

vace = VaceWanModel(
    vace_layers=vace_layers, vace_in_dim=vace_in_dim, patch_size=patch_size,
    has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim,
)
vace.eval()

vace_params = sum(p.numel() for p in vace.parameters())
print(f"2. VaceWanModel:         {vace_params:,} params, {len(vace_layers)} blocks, vace_in_dim={vace_in_dim}")
assert len(vace.vace_blocks) == len(vace_layers)
assert vace.vace_layers_mapping == {0: 0, 2: 1, 4: 2}

# --- 3. WanS2VModel ---
audio_dim = 384
num_audio_token = 4
audio_inject_layers = [0, 2]
cond_dim = 384

s2v = WanS2VModel(
    dim=dim, in_dim=in_dim, ffn_dim=ffn_dim, out_dim=out_dim,
    text_dim=text_dim, freq_dim=freq_dim, eps=eps,
    patch_size=patch_size, num_heads=num_heads, num_layers=num_layers,
    cond_dim=cond_dim, audio_dim=audio_dim, num_audio_token=num_audio_token,
    enable_adain=True, audio_inject_layers=audio_inject_layers,
    zero_timestep=True, add_last_motion=True, framepack_drop_mode="padd",
)
s2v.eval()

s2v_params = sum(p.numel() for p in s2v.parameters())
print(f"3. WanS2VModel:          {s2v_params:,} params, {num_layers} blocks, audio_inject={audio_inject_layers}")
assert hasattr(s2v, 'casual_audio_encoder')
assert hasattr(s2v, 'frame_packer')
assert hasattr(s2v, 'audio_injector')
assert hasattr(s2v, 'trainable_cond_mask')

print(f"\nAll three models instantiated and ready for model_fn_wan_video dispatch.")
print(f"  Patchify monkey-patch applied to WanModel (same fix as NB-07).")

## model_fn_wan_video: Three-Way Dispatch

Source: `diffsynth/pipelines/wan_video.py`, lines 1127-1214 (dispatch entry), lines 1307-1393 (VACE path)

The pipeline's `model_fn` selects the execution path based on available conditioning signals:

| Priority | Condition | Path | Model Called |
|----------|-----------|------|-------------|
| 1 (highest) | `audio_embeds is not None` | S2V | `model_fn_wans2v()` -- completely separate flow (NB-16) |
| 2 | `vace_context is not None` | VACE | Base DiT blocks + additive hint injection (NB-12) |
| 3 | Neither | Base | Pure DiT forward pass (NB-07) |

**Key insight:** The S2V check comes first (line 1201). If both `audio_embeds` and `vace_context` are provided, S2V takes priority. In practice, they are mutually exclusive control paths.

The remaining branches in production code (LongCat, sliding window tiler, USP sequence parallelism, TeaCache, animate adapter) are independent features outside this course's scope.

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, lines 1200-1214, 1222-1393
# Three-way dispatch demonstration

# Reduced spatial dims: num_frames=5, height=32, width=32
# -> T_lat=2 (temporal compression), H_lat=4, W_lat=4 (8x spatial compression)
# -> After patch_size=(1,2,2): f=2, h=2, w=2, S = 2*2*2 = 8
latents = torch.randn(1, 16, 2, 4, 4)
context_posi = torch.randn(1, 77, 512)  # positive prompt
context_nega = torch.randn(1, 77, 512)  # negative/null prompt
timestep_t = scheduler.timesteps[0].unsqueeze(0)  # first timestep

# ============================================================
# Helper: VACE/Base path (mirrors model_fn_wan_video lines 1222-1393)
# ============================================================
def model_fn_dispatch_demo(dit_model, vace_model, latents, timestep, context,
                           vace_ctx=None, vace_scale=1.0, audio_embeds_in=None,
                           s2v_model_in=None, s2v_audio=None, s2v_motion=None,
                           s2v_pose=None):
    """
    Simplified model_fn_wan_video dispatch.
    Source: diffsynth/pipelines/wan_video.py, lines 1127-1393
    """
    # Priority 1: S2V path (line 1201)
    if audio_embeds_in is not None:
        print("  -> S2V path (audio_embeds is not None)")
        # Delegate entirely to model_fn_wans2v -> s2v_model.forward()
        # Source: wan_video.py, lines 1202-1214
        out = s2v_model_in(
            latents=s2v_latents_input,
            timestep=timestep,
            context=context,
            audio_input=s2v_audio,
            motion_latents=s2v_motion,
            pose_cond=s2v_pose,
        )
        return out

    # Shared setup for paths 2 & 3 (lines 1222-1280)
    t = dit_model.time_embedding(
        sinusoidal_embedding_1d(dit_model.freq_dim, timestep).to(torch.float32))
    t_mod = dit_model.time_projection(t).unflatten(1, (6, dit_model.dim))
    context_emb = dit_model.text_embedding(context)
    x, (f, h, w) = dit_model.patchify(latents)
    print(f"     Shared setup: t_mod={t_mod.shape}, context_emb={context_emb.shape}, x={x.shape}")

    # RoPE frequencies
    freqs = torch.cat([
        dit_model.freqs[0][:f].view(f, 1, 1, -1).expand(f, h, w, -1),
        dit_model.freqs[1][:h].view(1, h, 1, -1).expand(f, h, w, -1),
        dit_model.freqs[2][:w].view(1, 1, w, -1).expand(f, h, w, -1)
    ], dim=-1).reshape(f * h * w, 1, -1).to(x.device)

    # Priority 2: VACE path (lines 1307-1375)
    if vace_ctx is not None:
        print("  -> VACE path (vace_context is not None)")
        # Pre-compute all VACE hints
        vace_hints = vace_model(x, vace_ctx, context_emb, t_mod, freqs)
        print(f"     VACE hints: {len(vace_hints)} hint tensors")
        # Block loop with additive injection
        for block_id, block in enumerate(dit_model.blocks):
            x = block(x, context_emb, t_mod, freqs)
            if block_id in vace_model.vace_layers_mapping:
                hint_idx = vace_model.vace_layers_mapping[block_id]
                x = x + vace_hints[hint_idx] * vace_scale
                print(f"     Block {block_id}: + VACE hint[{hint_idx}] * {vace_scale}")
            else:
                print(f"     Block {block_id}: no VACE hint")
    # Priority 3: Base path (lines 1334-1367)
    else:
        print("  -> Base path (no audio_embeds, no vace_context)")
        for block_id, block in enumerate(dit_model.blocks):
            x = block(x, context_emb, t_mod, freqs)
            print(f"     Block {block_id}: {x.shape}")

    # Head + unpatchify (lines 1383-1393)
    x = dit_model.head(x, t)
    x = dit_model.unpatchify(x, (f, h, w))
    return x

# ============================================================
# Demonstrate all three paths
# ============================================================
print("="*60)
print("PATH 1: Base DiT (no conditioning signals)")
print("="*60)
with torch.no_grad():
    out_base = model_fn_dispatch_demo(
        dit, vace, latents, timestep_t, context_posi)
assert out_base.shape == latents.shape, f"Base output {out_base.shape} != {latents.shape}"
print(f"     Output: {out_base.shape} == latents shape  [PASS]")

print(f"\n{'='*60}")
print("PATH 2: VACE (vace_context provided)")
print("="*60)
# Construct vace_context matching cell 6 output
vace_context = torch.randn(1, 96, 2, 4, 4)  # [B, vace_in_dim, T_lat, H_lat, W_lat]
with torch.no_grad():
    out_vace = model_fn_dispatch_demo(
        dit, vace, latents, timestep_t, context_posi,
        vace_ctx=vace_context)
assert out_vace.shape == latents.shape, f"VACE output {out_vace.shape} != {latents.shape}"
print(f"     Output: {out_vace.shape} == latents shape  [PASS]")

print(f"\n{'='*60}")
print("PATH 3: S2V (audio_embeds provided)")
print("="*60)
# S2V uses the WanS2VModel directly with its own input conventions
# S2V latents: [B, 16, num_frames, H, W] where H, W are pre-patchify spatial dims
# With s2v config from cell 12: patch_size=(1,2,2)
# S2V model expects raw latent-space dims (patchify happens inside the model)
# Use H=W=8 (minimal for patch_size 2) and F_content=4 (content) + 1 (ref) = 5 total
F_content_s2v = 4
s2v_H, s2v_W = 8, 8  # spatial dims (pre-patchify, must be divisible by patch_size)
s2v_latents_input = torch.randn(1, 16, F_content_s2v + 1, s2v_H, s2v_W)
print(f"  S2V latents: {s2v_latents_input.shape} (1 ref + {F_content_s2v} content frames)")

# S2V conditioning signals (matching cell 12 config: audio_dim=384, cond_dim=384)
s2v_audio_input = torch.randn(1, 25, 384, 16)  # [B, wav2vec_layers, audio_dim, T_audio]
s2v_motion_input = [torch.randn(16, 5, s2v_H, s2v_W)]  # motion frames
s2v_pose_input = torch.randn(1, 384, F_content_s2v, s2v_H, s2v_W)  # cond_dim=384
print(f"  audio_input:    {s2v_audio_input.shape}")
print(f"  motion_latents: {s2v_motion_input[0].shape}  (list of 1)")
print(f"  pose_cond:      {s2v_pose_input.shape}")

with torch.no_grad():
    out_s2v = model_fn_dispatch_demo(
        dit, vace, latents, timestep_t, context_posi,
        audio_embeds_in=s2v_audio_input,
        s2v_model_in=s2v,
        s2v_audio=s2v_audio_input,
        s2v_motion=s2v_motion_input,
        s2v_pose=s2v_pose_input)
assert out_s2v.shape == s2v_latents_input.shape, f"S2V output {out_s2v.shape} != {s2v_latents_input.shape}"
print(f"     Output: {out_s2v.shape} == s2v_latents shape  [PASS]")

print(f"\n{'='*60}")
print("Three-way dispatch demo complete.")
print(f"  Base:  {out_base.shape}")
print(f"  VACE:  {out_vace.shape}")
print(f"  S2V:   {out_s2v.shape}")

## Denoising Loop: VACE Path with Classifier-Free Guidance

Source: `diffsynth/pipelines/wan_video.py`, lines 290-314

The denoising loop calls `model_fn` at each timestep, applies CFG between positive and negative predictions, then updates latents via `scheduler.step`. We demonstrate the **VACE path** (5 steps per D-06) with `cfg_scale=5.0`.

**CFG formula:** `noise_pred = noise_pred_nega + cfg_scale * (noise_pred_posi - noise_pred_nega)`

This is the two-call variant (`cfg_merge=False`, the default). The batch-merged variant (`cfg_merge=True`) concatenates positive and negative into a single batch and splits after the forward pass.

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, lines 290-314
# VACE path denoising loop with classifier-free guidance

def vace_forward_pass(dit_model, vace_model, latents_in, timestep_in, context_in,
                      vace_ctx, vace_scale=1.0):
    """
    Single VACE forward pass (model_fn_wan_video, VACE branch).
    Source: diffsynth/pipelines/wan_video.py, lines 1222-1393
    """
    # Time embedding + projection
    t = dit_model.time_embedding(
        sinusoidal_embedding_1d(dit_model.freq_dim, timestep_in).to(torch.float32))
    t_mod = dit_model.time_projection(t).unflatten(1, (6, dit_model.dim))
    # Text embedding
    context_emb = dit_model.text_embedding(context_in)
    # Patchify
    x, (f, h, w) = dit_model.patchify(latents_in)
    # RoPE frequencies
    freqs = torch.cat([
        dit_model.freqs[0][:f].view(f, 1, 1, -1).expand(f, h, w, -1),
        dit_model.freqs[1][:h].view(1, h, 1, -1).expand(f, h, w, -1),
        dit_model.freqs[2][:w].view(1, 1, w, -1).expand(f, h, w, -1)
    ], dim=-1).reshape(f * h * w, 1, -1).to(x.device)
    # Pre-compute VACE hints
    vace_hints = vace_model(x, vace_ctx, context_emb, t_mod, freqs)
    # Block loop with additive VACE injection
    for block_id, block in enumerate(dit_model.blocks):
        x = block(x, context_emb, t_mod, freqs)
        if block_id in vace_model.vace_layers_mapping:
            x = x + vace_hints[vace_model.vace_layers_mapping[block_id]] * vace_scale
    # Head + unpatchify
    x = dit_model.head(x, t)
    x = dit_model.unpatchify(x, (f, h, w))
    return x

# Denoising loop
cfg_scale = 5.0
latents_vace = torch.randn(1, 16, 2, 4, 4)  # initial noise
vace_context = torch.randn(1, 96, 2, 4, 4)  # from VACE preprocessing (cell 6)

print("VACE Denoising Loop (5 steps, cfg_scale=5.0)")
print(f"Initial latents: {latents_vace.shape}")
print(f"VACE context:    {vace_context.shape}")
print()

with torch.no_grad():
    for progress_id, timestep in enumerate(scheduler.timesteps):
        timestep_tensor = timestep.unsqueeze(0)

        # Positive prediction (VACE path with real prompt)
        # Source: wan_video.py, line 301
        noise_pred_posi = vace_forward_pass(
            dit, vace, latents_vace, timestep_tensor, context_posi,
            vace_ctx=vace_context)

        # Negative prediction (VACE path with null/negative prompt)
        # Source: wan_video.py, line 306
        noise_pred_nega = vace_forward_pass(
            dit, vace, latents_vace, timestep_tensor, context_nega,
            vace_ctx=vace_context)

        # CFG formula (Source: wan_video.py, line 307)
        noise_pred = noise_pred_nega + cfg_scale * (noise_pred_posi - noise_pred_nega)

        # Scheduler step (Source: flow_match.py, lines 149-158)
        latents_vace = scheduler.step(
            noise_pred, scheduler.timesteps[progress_id], latents_vace)

        assert latents_vace.shape == torch.Size([1, 16, 2, 4, 4]), \
            f"Shape mismatch at step {progress_id}: {latents_vace.shape}"
        print(f"  Step {progress_id}: sigma={scheduler.sigmas[progress_id]:.4f} "
              f"-> noise_pred={noise_pred.shape} -> latents={latents_vace.shape}")

print(f"\nFinal VACE latents: {latents_vace.shape}")
print(f"Shape preserved through all 5 denoising steps: [PASS]")

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, lines 290-314 (S2V path)
# S2V denoising loop with classifier-free guidance

cfg_scale_s2v = 5.0

# S2V latents: [B, 16, num_frames, H, W] where H, W are spatial (pre-patchify)
# Using same dims as dispatch demo: F=5 (1 ref + 4 content), H=W=8
F_content_s2v = 4
s2v_H, s2v_W = 8, 8
latents_s2v = torch.randn(1, 16, F_content_s2v + 1, s2v_H, s2v_W)  # initial noise

# Conditioning signals (matching cell 12 config: audio_dim=384, cond_dim=384)
s2v_audio_input = torch.randn(1, 25, 384, 16)  # [B, wav2vec_layers, audio_dim, T_audio]
s2v_motion_input = [torch.randn(16, 5, s2v_H, s2v_W)]  # motion frames
s2v_pose_input = torch.randn(1, 384, F_content_s2v, s2v_H, s2v_W)  # cond_dim=384

# Fresh scheduler for S2V
scheduler_s2v = FlowMatchScheduler("Wan")
scheduler_s2v.set_timesteps(num_inference_steps=5, shift=5.0)

print("S2V Denoising Loop (5 steps, cfg_scale=5.0)")
print(f"Initial latents: {latents_s2v.shape}")
print(f"  (1 ref frame + {F_content_s2v} content frames, spatial {s2v_H}x{s2v_W})")
print()

with torch.no_grad():
    for progress_id, timestep in enumerate(scheduler_s2v.timesteps):
        timestep_tensor = timestep.unsqueeze(0)

        # Positive prediction (S2V path)
        # Source: wan_video.py, lines 1201-1214 -> model_fn_wans2v -> s2v.forward()
        noise_pred_posi = s2v(
            latents=latents_s2v,
            timestep=timestep_tensor,
            context=context_posi,
            audio_input=s2v_audio_input,
            motion_latents=s2v_motion_input,
            pose_cond=s2v_pose_input,
        )

        # Negative prediction (S2V path with null/negative prompt)
        # Source: wan_video.py, line 306
        noise_pred_nega = s2v(
            latents=latents_s2v,
            timestep=timestep_tensor,
            context=context_nega,
            audio_input=s2v_audio_input,
            motion_latents=s2v_motion_input,
            pose_cond=s2v_pose_input,
        )

        # CFG formula (Source: wan_video.py, line 307)
        noise_pred = noise_pred_nega + cfg_scale_s2v * (noise_pred_posi - noise_pred_nega)

        # Scheduler step (Source: flow_match.py, lines 149-158)
        latents_s2v = scheduler_s2v.step(
            noise_pred, scheduler_s2v.timesteps[progress_id], latents_s2v)

        assert latents_s2v.shape == torch.Size([1, 16, F_content_s2v + 1, s2v_H, s2v_W]), \
            f"Shape mismatch at step {progress_id}: {latents_s2v.shape}"
        print(f"  Step {progress_id}: sigma={scheduler_s2v.sigmas[progress_id]:.4f} "
              f"-> noise_pred={noise_pred.shape} -> latents={latents_s2v.shape}")

print(f"\nFinal S2V latents: {latents_s2v.shape}")
print(f"Shape preserved through all 5 denoising steps: [PASS]")

# Post-processing: WanVideoPostUnit_S2V
# Source: diffsynth/pipelines/wan_video.py, lines 895-903
print(f"\nPost-processing (WanVideoPostUnit_S2V):")
print(f"  Source: wan_video.py, lines 895-903")
print(f"  When drop_motion_frames=True (no real motion video): no-op")
print(f"  When motion video provided: torch.cat([motion_latents, latents[:,:,1:]], dim=2)")
print(f"  In this demo: drop_motion_frames=True -> output = latents unchanged")

## VAE Decode

Source: `diffsynth/pipelines/wan_video.py`, line 328; `diffsynth/models/wan_video_vae.py`, lines 1077-1121

After the denoising loop, the pipeline calls `vae.decode(latents)` to convert latents back to video frames:
- **Temporal expansion:** `T_out = T_lat * 4 - 3` (inverse of VAE temporal compression)
- **Spatial upsampling:** 8x (inverse of VAE 4-level spatial downsampling)
- **Channel mapping:** 16 latent channels -> 3 RGB channels

For full VAE decoder internals, see NB-10.

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 1077-1121
# VAE decode shape trace (shape arithmetic only -- see NB-10 for full decoder)

print("VAE Decode Shape Trace")
print("=" * 50)

# VACE path latents: from the denoising loop above
T_lat_vace, H_lat_vace, W_lat_vace = 2, 4, 4  # from latents_vace
print(f"\nVACE path VAE decode:")
print(f"  Input latents:  [1, 16, {T_lat_vace}, {H_lat_vace}, {W_lat_vace}]")
T_out_vace = T_lat_vace * 4 - 3  # temporal expansion formula
H_out_vace = H_lat_vace * 8       # 8x spatial upsampling
W_out_vace = W_lat_vace * 8
print(f"  Output video:   [1, 3, {T_out_vace}, {H_out_vace}, {W_out_vace}]")
print(f"  Temporal: T_lat={T_lat_vace} -> T_out={T_out_vace}  (T_lat * 4 - 3)")
print(f"  Spatial:  {H_lat_vace}x{W_lat_vace} -> {H_out_vace}x{W_out_vace}  (8x upsampling)")
print(f"  Channels: 16 latent -> 3 RGB")
assert T_out_vace == 5, f"Expected T_out=5, got {T_out_vace}"  # matches num_frames=5

# S2V path latents: from the S2V denoising loop
# S2V output: [1, 16, 5, 8, 8] (ref frame included)
T_lat_s2v, H_lat_s2v, W_lat_s2v = F_content_s2v + 1, s2v_H, s2v_W  # 5, 8, 8
print(f"\nS2V path VAE decode:")
print(f"  Input latents:  [1, 16, {T_lat_s2v}, {H_lat_s2v}, {W_lat_s2v}]")
T_out_s2v = T_lat_s2v * 4 - 3
H_out_s2v = H_lat_s2v * 8
W_out_s2v = W_lat_s2v * 8
print(f"  Output video:   [1, 3, {T_out_s2v}, {H_out_s2v}, {W_out_s2v}]")
print(f"  Temporal: T_lat={T_lat_s2v} -> T_out={T_out_s2v}  (T_lat * 4 - 3)")
print(f"  Spatial:  {H_lat_s2v}x{W_lat_s2v} -> {H_out_s2v}x{W_out_s2v}  (8x upsampling)")
print(f"  Channels: 16 latent -> 3 RGB")
assert T_out_s2v == 17, f"Expected T_out=17, got {T_out_s2v}"  # 5*4 - 3 = 17

# Verify with torch tensors (shape only, no real VAE)
decoded_vace = torch.randn(1, 3, T_out_vace, H_out_vace, W_out_vace)
decoded_s2v = torch.randn(1, 3, T_out_s2v, H_out_s2v, W_out_s2v)
assert decoded_vace.shape == torch.Size([1, 3, 5, 32, 32])
assert decoded_s2v.shape == torch.Size([1, 3, 17, 64, 64])
print(f"\nDecoded video shapes verified:")
print(f"  VACE: {decoded_vace.shape}")
print(f"  S2V:  {decoded_s2v.shape}")

## Shape Trace Summary

| Stage | Tensor | VACE Path Shape | S2V Path Shape | Source |
|-------|--------|-----------------|----------------|--------|
| Initial noise | `latents` | `[1, 16, 2, 4, 4]` | `[1, 16, 5, 8, 8]` | NoiseInitializer |
| VACE context | `vace_context` | `[1, 96, 2, 4, 4]` | N/A | WanVideoUnit_VACE |
| S2V audio | `audio_input` | N/A | `[1, 25, 384, 16]` | WanVideoUnit_S2V |
| S2V motion | `motion_latents` | N/A | `[16, 5, 8, 8]` (list) | WanVideoUnit_S2V |
| S2V pose | `pose_cond` | N/A | `[1, 384, 4, 8, 8]` | WanVideoUnit_S2V |
| Prompt (positive) | `context_posi` | `[1, 77, 512]` | `[1, 77, 512]` | PromptEmbedder |
| Prompt (negative) | `context_nega` | `[1, 77, 512]` | `[1, 77, 512]` | PromptEmbedder |
| Sigma schedule | `sigmas` | `[5]` | `[5]` | FlowMatchScheduler |
| Noise prediction | `noise_pred` | `[1, 16, 2, 4, 4]` | `[1, 16, 5, 8, 8]` | model_fn dispatch |
| CFG combined | `noise_pred` | `[1, 16, 2, 4, 4]` | `[1, 16, 5, 8, 8]` | Inline CFG formula |
| After scheduler step | `latents` | `[1, 16, 2, 4, 4]` | `[1, 16, 5, 8, 8]` | FlowMatchScheduler.step |
| VAE decode | `video` | `[1, 3, 5, 32, 32]` | `[1, 3, 17, 64, 64]` | WanVideoVAE.decode |

**Key shapes:**
- **VACE latents** are compact: `[1, 16, 2, 4, 4]` = 512 values per step (from 32x32 video, 5 frames)
- **S2V latents** include the reference frame: `[1, 16, 5, 8, 8]` = 5120 values per step (from 64x64 video, 5 frames)
- **VAE temporal formula** `T_out = T_lat * 4 - 3`: VACE 2->5 frames, S2V 5->17 frames
- **CFG doubles compute** (not memory): two forward passes per step with `cfg_merge=False`

## Key Architectural Insights

1. **Unified interface, divergent paths:** `model_fn_wan_video` presents a single entry point, but S2V immediately delegates to a completely different function (`model_fn_wans2v`) with its own forward pass, while VACE augments the base DiT path with additive hints.

2. **Preprocessing absorbs complexity:** The PipelineUnit chain handles all conditioning signal preparation (VAE encoding, audio encoding, mask construction) BEFORE the denoising loop. The loop itself is simple: forward pass + CFG + scheduler step.

3. **CFG is architecture-agnostic:** The same `noise_pred = nega + scale * (posi - nega)` formula works identically for VACE, S2V, and base paths -- it only cares about the noise prediction output shape.

4. **Scheduler step is trivial:** `prev = sample + output * (sigma_next - sigma)` -- the mathematical sophistication is in the sigma schedule (the `shift` parameter), not the step formula.

## Summary

### Key Takeaways
- **PipelineUnit chain** preprocesses all conditioning signals before the denoising loop: VACE constructs `vace_context`, S2V constructs `audio_embeds` + `motion_latents` + `pose_latents`
- **Three-way dispatch** in `model_fn_wan_video`: S2V (highest priority, via `audio_embeds`), VACE (additive hints on base DiT), or pure base path
- **Denoising loop** runs N steps (5 in demo, 50 in production): model forward -> CFG -> scheduler step, with shape preserved throughout
- **FlowMatchScheduler** provides shifted sigma schedule and step formula: `prev = sample + output * (sigma_next - sigma)`
- **VAE decode** converts latents to video: temporal expansion (`T_lat * 4 - 3`), 8x spatial upsampling, 16->3 channel mapping

### Source References

| Symbol | Location |
|--------|----------|
| WanVideoPipeline.__call__ | `diffsynth/pipelines/wan_video.py`, line 178 |
| model_fn_wan_video | `diffsynth/pipelines/wan_video.py`, line 1127 |
| model_fn_wans2v | `diffsynth/pipelines/wan_video.py`, line 1426 |
| FlowMatchScheduler | `diffsynth/diffusion/flow_match.py`, line 5 |
| PipelineUnit | `diffsynth/diffusion/base_pipeline.py`, line 13 |
| WanVideoUnit_VACE | `diffsynth/pipelines/wan_video.py`, line 621 |
| WanVideoUnit_S2V | `diffsynth/pipelines/wan_video.py`, line 811 |

### Course Complete
You have now traced the complete WAN 2.2 architecture from primitives (NB-01) through the full pipeline (NB-17). The 17 notebooks cover: DiT primitives (5), DiT assembly (2), VAE (3), VACE control (2), S2V pipeline (4), and integration (1).

## Exercises

### Exercise 1 -- cfg_merge vs Two-Call CFG
What happens if you set `cfg_merge=True`? Instead of two separate `model_fn` calls, the positive and negative prompts are batch-concatenated (`batch_size=2`) and a single forward pass is made. After the call, `noise_pred.chunk(2, dim=0)` splits the output. What are the performance tradeoffs?

**Hint:** One forward pass with `batch_size=2` vs two forward passes with `batch_size=1`. Memory vs compute. See `wan_video.py`, lines 302-304.

### Exercise 2 -- Sigma Schedule Shift Parameter
The `shift=5.0` parameter controls how aggressively the sigma schedule is shifted toward higher noise levels. Try `shift=1.0` (no shift) and `shift=10.0` (aggressive shift). How does the sigma distribution change? What does this mean for denoising behavior?

**Hint:** `sigmas_shifted = shift * sigmas / (1 + (shift - 1) * sigmas)`. Plot both schedules. Higher shift = more time spent at high noise levels = better global structure, less fine detail.

### Exercise 3 -- Production Scale
In production, WAN 2.2 uses: 40 DiT layers, 20 VACE blocks, 720x480 video at 81 frames (T_lat=21), 50 denoising steps. How many total model forward passes happen for a VACE generation with `cfg_merge=False`? What about with `cfg_merge=True`?

**Answer:** Without merge: 50 steps x 2 calls/step = 100 DiT forward passes + 50 VACE forward passes (VACE is called inside each model_fn invocation). With merge: 50 DiT forward passes (batch=2) + 50 VACE forward passes. In both cases, the VACE hint pre-computation happens once per model_fn call.